In [1]:
import pandas as pd
import wm_unit as wmu
from datetime import datetime
from datetime import timedelta

def next_turn(current_time):
    current_time += timedelta(minutes=150)
    print(current_time)
    return current_time


def create_ground_unit(
    unit_id,
    side="blue",
    inf_df=None,
    armor_df=None,
):
    """
    Универсальный конструктор ground unit

    Варианты:
    1) inf_df only       -> infantry
    2) armor_df only     -> armor
    3) inf_df + armor_df -> mech
    """

    # ---------- validation ----------
    if inf_df is None and armor_df is None:
        raise ValueError("Need inf_df and/or armor_df")

    # ---------- INF PREP ----------
    if inf_df is not None:
        inf_df = inf_df.copy().reset_index(drop=True)

        if "power" not in inf_df.columns:
            inf_df["power"] = 1

        if "inf_kills" not in inf_df.columns:
            inf_df["inf_kills"] = 0

        if "apc_kills" not in inf_df.columns:
            inf_df["apc_kills"] = 0


    # ---------- ARMOR PREP ----------
    if armor_df is not None:
        armor_df = armor_df.copy().reset_index(drop=True)

        if "power" not in armor_df.columns:
            armor_df["power"] = 8

        if "type" not in armor_df.columns:
            armor_df["type"] = "apc"

        if "transport" not in armor_df.columns:
            armor_df["transport"] = 9

        if "inf_kills" not in armor_df.columns:
            armor_df["inf_kills"] = 0

        if "apc_kills" not in armor_df.columns:
            armor_df["apc_kills"] = 0


    # ==================================================
    # 1 ARM UNIT
    # ==================================================
    if inf_df is None and armor_df is not None:

        arm = wmu.Unit(
            unit_id=unit_id,
            overall_type="arm",
            personal_type="arm",
            alive_df=armor_df,
            cas_df=pd.DataFrame(columns=armor_df.columns),
            side=side,
        )

        arm.size = "ARMOR"

        return arm


    # ==================================================
    # 2 INF UNIT
    # ==================================================
    if inf_df is not None and armor_df is None:

        inf = wmu.Unit(
            unit_id=unit_id,
            overall_type="inf",
            personal_type="inf",
            alive_df=inf_df,
            cas_df=pd.DataFrame(columns=inf_df.columns),
            side=side,
        )

        return inf


    # ==================================================
    # 3 MECH UNIT
    # ==================================================
    armor_part = wmu.Unit(
        unit_id=unit_id + "_ARM",
        overall_type="arm",
        personal_type="arm",
        alive_df=armor_df,
        cas_df=pd.DataFrame(columns=armor_df.columns),
        side=side,
    )

    armor_part.size = "ARMOR"

    mech = wmu.Unit(
        unit_id=unit_id,
        overall_type="inf",
        personal_type="inf",
        alive_df=inf_df,
        cas_df=pd.DataFrame(columns=inf_df.columns),
        side=side,
    )

    mech.armor_part = armor_part
    mech.update_current_type()   # станет mech

    return mech






def run_ground_battle(
    attacker,
    defender,
    current_time=None,
    attacker_cover=0,
    defender_cover=0,
    attacker_elevation=0,
    defender_elevation=0,
    distance=0,
    defender_returns_fire=True,
    attacker_berserk=False,
    defender_berserk=False,
    armor_is_moving=False,
    armor_is_far=False,
):
    """
    Запускает ОДИН ground battle между двумя Unit.

    defender_returns_fire=True  -> обоюдный бой
    defender_returns_fire=False -> односторонняя атака без ответа
    """

    global ground_logs

    if current_time is None:
        current_time = datetime.now()

    context = BattleContext(
        attacker_cover=attacker_cover,
        defender_cover=defender_cover,
        attacker_elevation=attacker_elevation,
        defender_elevation=defender_elevation,
        distance=distance,
        defender_returns_fire=defender_returns_fire,
        attacker_berserk=attacker_berserk,
        defender_berserk=defender_berserk,
        armor_is_moving=armor_is_moving,
        armor_is_far=armor_is_far,
    )

    result = engine.resolve_engagement(
        attacker=attacker,
        defender=defender,
        context=context,
        logs=ground_logs,
        current_time=current_time,
    )

    ground_logs = result.logs

    return result


from datetime import datetime
import pandas as pd
from ground_engine import GroundEngine, BattleContext

engine = GroundEngine()

GROUND_LOG_COLUMNS = [
    "current_time",
    "initiator",
    "log_blue_id",
    "log_blue_type",
    "log_blue_inf_force",
    "log_blue_arm_force",
    "log_blue_cas_inf",
    "log_blue_cas_armor",
    "log_red_id",
    "log_red_type",
    "log_red_inf_force",
    "log_red_arm_force",
    "log_red_cas_inf",
    "log_red_cas_armor",
    "log_attack_type",
    "log_result",
]

ground_logs = pd.DataFrame(columns=GROUND_LOG_COLUMNS)


In [2]:

from force_manager import ForceManager
from ground_engine import GroundEngine, BattleContext
import wm_unit as wmu

# ------------------------------------------------------------
# FILES
# ------------------------------------------------------------
PLAYER_FILE = "player_brigade.xlsx"
ENEMY_FILE = "enemy_brigade.xlsx"

player_manager = ForceManager.from_excel(PLAYER_FILE)
enemy_manager = ForceManager.from_excel(ENEMY_FILE)



In [3]:
apc1 = player_manager.armor_df.iloc[:9]
appp = player_manager.armor_df.iloc[9:15]
app3 = player_manager.armor_df.iloc[15:8]

In [4]:
pc1 = 0
ppp = 0
pp3 = 0

pp = [[pc1,['B2-C1'],'company_uid','pc1',apc1],
[ppp,['B1-C2-P1','B1-C2-P2'],'platoon_uid','ppp',appp],
[pp3,['B1-C2-P3'],'platoon_uid','pp3',app3]]

counter = 0
for p in pp:
    inf_df = player_manager.master_df[(player_manager.master_df[p[2]].isin(p[1]))&
                                     (player_manager.master_df['status']=='alive')]

    pp[counter][0] = create_ground_unit(
        unit_id=p[3],
        side="blue",
        inf_df=inf_df,
        armor_df=p[4])
    counter+=1
    
pc1 = pp[0][0]
ppp = pp[1][0]
pp3 = pp[2][0]


In [6]:
inf_df = enemy_manager.master_df[
    enemy_manager.master_df['platoon_uid'].isin(['B6-C1-P1'])&
    (enemy_manager.master_df['status']=='alive')]

ep1  = create_ground_unit(
    unit_id='ep1',
    side="red",
    inf_df=inf_df)


In [8]:
START_TIME = datetime(2026, 4, 6, 6, 0)

In [38]:
battle_result = run_ground_battle(
    attacker=ppp,
    defender=ec1,
    current_time=str(START_TIME),
    attacker_cover=0,
    defender_cover=1,
    attacker_elevation=0,
    defender_elevation=0,
    distance=0,
    defender_returns_fire=True,   # False = атака без ответа
)



In [43]:
pc1.alive_df[pc1.alive_df['squad_uid']=='B2-C1-P1-S2']

,global_soldier_id,soldier_id,squad_id,platoon_id,company_id,battalion_id,battalion_uid,company_uid,platoon_uid,squad_uid,soldier_uid,status,power,side,inf_kills,apc_kills
14,393,1,2,1,1,2,B2,B2-C1,B2-C1-P1,B2-C1-P1-S2,B2-C1-P1-S2-U1,alive,1,NaN,0,0
15,394,2,2,1,1,2,B2,B2-C1,B2-C1-P1,B2-C1-P1-S2,B2-C1-P1-S2-U2,alive,1,NaN,1,0
16,395,3,2,1,1,2,B2,B2-C1,B2-C1-P1,B2-C1-P1-S2,B2-C1-P1-S2-U3,alive,1,NaN,2,0
17,396,4,2,1,1,2,B2,B2-C1,B2-C1-P1,B2-C1-P1-S2,B2-C1-P1-S2-U4,alive,1,NaN,1,0
18,397,5,2,1,1,2,B2,B2-C1,B2-C1-P1,B2-C1-P1-S2,B2-C1-P1-S2-U5,alive,1,NaN,0,0
19,398,6,2,1,1,2,B2,B2-C1,B2-C1-P1,B2-C1-P1-S2,B2-C1-P1-S2-U6,alive,1,NaN,0,0
20,399,7,2,1,1,2,B2,B2-C1,B2-C1-P1,B2-C1-P1-S2,B2-C1-P1-S2-U7,alive,1,NaN,1,0
21,400,8,2,1,1,2,B2,B2-C1,B2-C1-P1,B2-C1-P1-S2,B2-C1-P1-S2-U8,alive,1,NaN,1,0
22,401,9,2,1,1,2,B2,B2-C1,B2-C1-P1,B2-C1-P1-S2,B2-C1-P1-S2-U9,alive,1,NaN,0,0
23,402,10,2,1,1,2,B2,B2-C1,B2-C1-P1,B2-C1-P1-S2,B2-C1-P1-S2-U10,alive,1,NaN,1,0


In [28]:
START_TIME = next_turn(START_TIME)

2026-04-06 18:30:00


In [83]:
for podr in [pc1,ppp,pp3]:
    cas = podr.cas_df['global_soldier_id']
    podr.cas_df['status'] = 'killed'
    player_manager.master_df = player_manager.master_df[~player_manager.master_df['global_soldier_id'].isin(cas)]
    player_manager.master_df = player_manager.master_df.append(podr.cas_df)
    
    alive = podr.alive_df['global_soldier_id']
    player_manager.master_df = player_manager.master_df[~player_manager.master_df['global_soldier_id'].isin(alive)]
    player_manager.master_df = player_manager.master_df.append(podr.alive_df)

<ipython-input-83-5b54000122be>:5: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  player_manager.master_df = player_manager.master_df.append(podr.cas_df)
<ipython-input-83-5b54000122be>:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  player_manager.master_df = player_manager.master_df.append(podr.alive_df)
<ipython-input-83-5b54000122be>:5: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  player_manager.master_df = player_manager.master_df.append(podr.cas_df)
<ipython-input-83-5b54000122be>:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  player_manager.master_df = player_manager.master_df.append(podr.alive_df)
<ipython-input-83-5b54000122

In [81]:
for podr in [ep1,ec1,ec2]:
    cas = podr.cas_df['global_soldier_id']
    podr.cas_df['status'] = 'killed'
    enemy_manager.master_df = enemy_manager.master_df[~enemy_manager.master_df['global_soldier_id'].isin(cas)]
    enemy_manager.master_df = enemy_manager.master_df.append(podr.cas_df)
    
    alive = podr.alive_df['global_soldier_id']
    enemy_manager.master_df = enemy_manager.master_df[~enemy_manager.master_df['global_soldier_id'].isin(alive)]
    enemy_manager.master_df = enemy_manager.master_df.append(podr.alive_df)

<ipython-input-81-a414a6b7e3fa>:5: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  enemy_manager.master_df = enemy_manager.master_df.append(podr.cas_df)
<ipython-input-81-a414a6b7e3fa>:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  enemy_manager.master_df = enemy_manager.master_df.append(podr.alive_df)
<ipython-input-81-a414a6b7e3fa>:5: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  enemy_manager.master_df = enemy_manager.master_df.append(podr.cas_df)
<ipython-input-81-a414a6b7e3fa>:9: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  enemy_manager.master_df = enemy_manager.master_df.append(podr.alive_df)
<ipython-input-81-a414a6b7e3fa>:5: F

In [85]:
player_manager.save_to_excel(PLAYER_FILE)
enemy_manager.save_to_excel(ENEMY_FILE)

In [51]:
ground_logs.to_csv('saved\\0 LOGS-d1-b2.xlsx',index=False)